# PP434 portfolio: Coding challenges

All Python data cleaning and wrangling for the coding challenges is in this notebook.

**Folder convention** (paths are relative to this notebook, i.e. the `portfolio/` folder):
- `data/raw/`: files exactly as downloaded (never edited by hand)
- `data/clean/`: tidy outputs written by this notebook; the charts read these via their GitHub *raw* URL
- `charts/`: Vega-Lite `.json` specs embedded in `index.html`

Re-running the notebook from top to bottom rebuilds every file in `data/clean/`.

In [1]:
# Setup
import os
import pandas as pd

raw_path = os.path.join("data", "raw")
clean_path = os.path.join("data", "clean")
os.makedirs(clean_path, exist_ok=True)

## CC1. Hosting

## CC2. Building

Two German macro series from FRED:

| File | FRED series | Description | Frequency |
|---|---|---|---|
| `ger_bunds.csv` | [IRLTLT01DEM156N](https://fred.stlouisfed.org/series/IRLTLT01DEM156N) | Long-term (10-year) government bond yield, % | Monthly |
| `ger_inflation.csv` | [FPCPITOTLZGDEU](https://fred.stlouisfed.org/series/FPCPITOTLZGDEU) | CPI inflation, annual % | Annual |

### Setup: Tidy FRED data downloads

Defining a function that cleans FRED data: rename columns to `date,value`, keep dates in `yyyy-mm-dd` format and drop any missing observations.

In [2]:
def clean_fred(raw_file, clean_file):
    df = pd.read_csv(os.path.join(raw_path, raw_file))
    series_id = df.columns[1]
    df = df.rename(columns={"observation_date": "date", series_id: "value"})

    df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["value"]).sort_values("date")
    df["value"] = df["value"].round(3)

    df.to_csv(os.path.join(clean_path, clean_file), index=False)
    print(f"{raw_file} ({series_id}) -> {clean_file}: {len(df)} rows, {df['date'].min()} to {df['date'].max()}")
    return df

Apply functions to raw FRED data sets and save these to `clean_path` for visualisation in VegaLite.

In [3]:
bunds = clean_fred("ger_bunds.csv", "cc2_ger_bunds.csv")
bunds.head()

ger_bunds.csv (IRLTLT01DEM156N) -> cc2_ger_bunds.csv: 121 rows, 2016-10-01 to 2026-10-01


,date,value
0,2016-10-01,0.002
1,2016-11-01,0.189
2,2016-12-01,0.252
3,2017-01-01,0.251
4,2017-02-01,0.258


In [4]:
inflation = clean_fred("ger_inflation.csv", "cc2_ger_inflation.csv")
inflation.head()

ger_inflation.csv (FPCPITOTLZGDEU) -> cc2_ger_inflation.csv: 66 rows, 1960-01-01 to 2025-01-01


,date,value
0,1960-01-01,1.537
1,1961-01-01,2.294
2,1962-01-01,2.843
3,1963-01-01,2.967
4,1964-01-01,2.336


## CC3. Policy debate

**Policy debate:** As Winter approaches, how should Germany secure gas supply amid rising prices?

Two data sets:

| File | FRED series | Description | Frequency |
|---|---|---|---|
| `ger_gas.csv` | [Bundesnetzagentur](https://www.bundesnetzagentur.de/DE/Gasversorgung/aktuelle_gasversorgung/_svg/Gasspeicher_Fuellstand/Speicherfuellstand.html) | Gas storage levels, % | Daily |
| `eu_gas_price.csv` | [FRED](https://fred.stlouisfed.org/series/PNGASEUUSDM) | Global price of Natural gas, EU, in US Dollar | Monthly |

### First graph: Gas storage levels in Germany
Data: German gas storage levels (`ger_gas.csv`)

The raw file comes in German format and needs four fixes:
1. **Delimiter and decimals:** columns are separated by `;` and decimals use `,` (e.g. `29,14`).
2. **No year in the dates:** the first column only holds day and month (`01.04.`).
3. **Wide layout:** each gas year (1 April to 31 March) is its own column: `04/2025-03/2026` and `04/2026-03/2027`.
4. **Empty future days:** the current gas year has no values yet for the rest of the winter.

Output: a tidy file with one row per day and two columns, `date` (`yyyy-mm-dd`) and `gas_storage_pct`.

In [5]:
# Read the German-format CSV
gas = pd.read_csv(os.path.join(raw_path, "ger_gas.csv"), sep=";", decimal=",")

# Rename columns: first column = day and month, the others = start year of each gas year
gas.columns = ["day_month", "2025", "2026"]   # 04/2025-03/2026 and 04/2026-03/2027

# Wide -> long
gas = gas.melt(id_vars="day_month", var_name="gas_year", value_name="gas_storage_pct")

# Add the calendar year: April-December fall in the gas year's start year, January-March in the next year
day_month = gas["day_month"].str.strip().str.rstrip(".")
month = day_month.str[3:5].astype(int)
year = gas["gas_year"].astype(int) + (month < 4).astype(int)
gas["date"] = pd.to_datetime(day_month + "." + year.astype(str), format="%d.%m.%Y")

# Drop days without data, keep the two final columns, sort by date
gas = gas.dropna(subset=["gas_storage_pct"])[["date", "gas_storage_pct"]].sort_values("date")
gas["date"] = gas["date"].dt.strftime("%Y-%m-%d")

# Save cleaned csv to clean_path
gas.to_csv(os.path.join(clean_path, "cc3_ger_gas_storage.csv"), index=False)
gas.head()

,date,gas_storage_pct
0,2025-04-01,29.14
1,2025-04-02,29.21
2,2025-04-03,29.26
3,2025-04-04,29.36
4,2025-04-05,29.53


Data: Weekly average temperature in Germany `temp.csv` – to plot against gas storage levels.

In [67]:
temp = pd.read_csv(os.path.join(raw_path, "temp.csv"), sep=";", decimal=",")
temp = temp.rename(columns={temp.columns[0]: "week"})   # header of first column is blank

# Keep the week number and the years covered by the gas storage data
temp = temp[["week", "2025", "2026"]]

# Wide -> long
temp = temp.melt(id_vars="week", var_name="year", value_name="temp_c")

# Create ISO dates
temp["date"] = pd.to_datetime(
    temp["year"] + "-W" + temp["week"].astype(str).str.zfill(2) + "-1",
    format="%G-W%V-%u",
)

# Same layout as the gas file: date (yyyy-mm-dd) + value, no gaps, sorted by date
temp = temp.dropna(subset=["temp_c"])[["date", "temp_c"]].sort_values("date")
temp["date"] = temp["date"].dt.strftime("%Y-%m-%d")

# Save cleaned csv to clean_path
temp.to_csv(os.path.join(clean_path, "cc3_ger_temp.csv"), index=False)
temp.head()

,date,temp_c
0,2025-06-30,21.9
1,2025-07-07,16.7
2,2025-07-14,18.5
3,2025-07-21,18.6
4,2025-07-28,16.2


### Second graph: EU gas prices
Data: Global gas prices, EU (`eu_gas_price.csv`)

Apply `clean_fred()` function defined in CC2 to clean raw FRED data and save it in `clean_path`.

In [7]:
gas_price = clean_fred("eu_gas_price.csv", "cc3_eu_gas_price.csv")
gas_price.head()

eu_gas_price.csv (PNGASEUUSDM) -> cc3_eu_gas_price.csv: 417 rows, 1992-01-01 to 2026-09-01


,date,value
0,1992-01-01,2.38
1,1992-02-01,2.38
2,1992-03-01,2.38
3,1992-04-01,2.34
4,1992-05-01,2.34
